In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

## Xử lý data

### 1. Đọc dữ liệu

In [18]:
df = pd.read_csv("data/data.csv")
print(df.shape)
print(df.dtypes)
df.head()


(1074, 7)
Brand           object
Series          object
Version         object
Max Tension     object
Technologies    object
Origin          object
Price            int64
dtype: object


,Brand,Series,Version,Max Tension,Technologies,Origin,Price
0,Yonex,Giá Rẻ,NaN,NaN,NaN,Japan,638000
1,Yonex,PJB (Hàn Quốc),NaN,NaN,NaN,Japan,1733500
2,Yonex,NaN,Pro Kennex,NaN,EXTRA SLIM SHAFT,Japan,598000
3,Yonex,The 3rd,Game,NaN,"ISOMETRIC, SOLID FEEL CORE",Japan,650000
4,Yonex,Arcsaber 0 Clear,NaN,30 lbs,"ISOMETRIC, AERO-BOX FRAME, BUILT-IN T-JOINT",Japan,579000


### 2. Kiểm tra dữ liệu mất (NaN)

In [23]:
missing = df.isnull().sum()
print(missing[missing>0])
print(df.isnull().mean().sort_values(ascending=False) * 100)


Series            8
Version         864
Max Tension     180
Technologies     33
dtype: int64
Version         80.446927
Max Tension     16.759777
Technologies     3.072626
Series           0.744879
Brand            0.000000
Origin           0.000000
Price            0.000000
dtype: float64


In [26]:
# Đếm số công nghệ trong mỗi ô
def count_tech(val):
    if pd.isna(val) or val == "":  # ô trống → 0
        return 0
    return len(val.split(","))     # đếm số phần tử ngăn cách bởi dấu phẩy

df["num_technologies"] = df["Technologies"].apply(count_tech)

print(df["num_technologies"].value_counts())

num_technologies
5     183
6     176
7     142
4      91
8      79
3      74
2      67
9      62
11     49
10     46
1      36
0      33
12     20
13     12
14      4
Name: count, dtype: int64


### 3. Chọn feature & xử lý missing

In [27]:
FEATURES = ["Brand","Series","Version","Max Tension","num_technologies","Origin"]
TARGET   = "Price"
df_model = df[FEATURES + [TARGET]].copy()

for col in FEATURES:
    if df_model[col].dtype == object:
        mode_val = df_model[col].mode()[0]  #gia tri xuat hien nhieu nhat
        df_model[col] = df_model[col].fillna(mode_val)  
    else:
        df_model[col] = df_model[col].fillna(df_model[col].median())

print(df_model.isnull().sum())

Brand               0
Series              0
Version             0
Max Tension         0
num_technologies    0
Origin              0
Price               0
dtype: int64


### 4. Label encoding

In [30]:
label_maps = {}     
for col in FEATURES:
    if df_model[col].dtypes == object:
        codes,uniques = pd.factorize(df_model[col])     #code = [0,1,0,2,1]  , uniques = ["Yonex", "Victor", "Bubadu"]  ← bảng tra cứu
        df_model[col] = codes
        label_maps[col] = uniques     #  Lưu bảng tra cứu lại để sau này biết 0 → Yonex, 1 → Victor...
        print(f"{col}: {dict(enumerate(uniques[:5]))} ...")
# X và y cuối cùng
X = df_model[FEATURES].values.astype(float)  # shape (754, 6)
y = df_model[TARGET].values.astype(float)    # shape (754,)
print(f"X: {X.shape}, y: {y.shape}")
print(f"Giá: min={y.min():,.0f} | max={y.max():,.0f} | mean={y.mean():,.0f} VNĐ")
print(df_model.head())

X: (1074, 6), y: (1074,)
Giá: min=349,000 | max=13,500,000 | mean=2,191,205 VNĐ
   Brand  Series  Version  Max Tension  num_technologies  Origin    Price
0      0       0        0            0                 0       0   638000
1      0       1        0            0                 0       0  1733500
2      0       2        1            0                 1       0   598000
3      0       3        2            0                 2       0   650000
4      0       4        0            1                 3       0   579000


### 5. Chia tập train/test

In [29]:
def train_test_split(X, y, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    n_test = int(len(y) * test_ratio)
    return X[idx[n_test:]], X[idx[:n_test]], y[idx[n_test:]], y[idx[:n_test]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_ratio=0.2)
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

Train: 860 | Test: 214
